In [3]:
import pandas as pd
import os
df = pd.read_csv("../data/raw/india_job_market_2024_2026.csv")
df.shape

(5000, 17)

In [4]:
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print(df["Experience_Level"].value_counts())

Missing values: 0
Duplicate rows: 0
Experience_Level
Mid (3-6 yrs)        1378
Junior (1-3 yrs)     1256
Fresher (0-1 yr)     1022
Senior (6-10 yrs)     831
Lead (10+ yrs)        513
Name: count, dtype: int64


In [5]:
text_cols = ["Job_Title", "Company", "Company_Type", "City", "Experience_Level"]
for col in text_cols:
    df[col] = df[col].str.strip()

df["Date_Posted"] = pd.to_datetime(df["Date_Posted"])

In [6]:
df["Skills_List"] = df["Skills_Required"].str.split(",").apply(
    lambda skills: [s.strip() for s in skills]
)
df[["Skills_Required", "Skills_List"]].head()

,Skills_Required,Skills_List
0,"Kotlin, Java, REST APIs","[Kotlin, Java, REST APIs]"
1,"Selenium, Manual Testing, Postman, API Testing...","[Selenium, Manual Testing, Postman, API Testin..."
2,"JIRA, Excel, Power BI","[JIRA, Excel, Power BI]"
3,"Penetration Testing, Python, Ethical Hacking, ...","[Penetration Testing, Python, Ethical Hacking,..."
4,"Docker, REST APIs, AWS, PostgreSQL","[Docker, REST APIs, AWS, PostgreSQL]"


In [7]:
entry_levels = ["Fresher", "Junior"]
df["Is_Entry_Level"] = df["Experience_Level"].str.startswith(tuple(entry_levels))
df["Is_Entry_Level"].value_counts()

Is_Entry_Level
False    2722
True     2278
Name: count, dtype: int64

In [8]:
df["Is_Analyst"] = df["Job_Title"].str.contains("Analyst")
df["Is_Analyst"].value_counts()

Is_Analyst
False    4452
True      548
Name: count, dtype: int64

In [9]:
df["Is_Remote"] = df["City"] == "Remote"
df["Is_Remote"].value_counts()

Is_Remote
False    3008
True     1992
Name: count, dtype: int64

In [10]:
target = df[df["Is_Entry_Level"] & df["Is_Analyst"]]
print(len(target))
target["Job_Title"].value_counts()

230


Job_Title
Data Analyst             94
Business Analyst         89
Cybersecurity Analyst    47
Name: count, dtype: int64

In [12]:
df["Salary_LPA"].describe()

count    5000.000000
mean       19.829440
std        18.136741
min         0.800000
25%         6.800000
50%        13.600000
75%        25.600000
max       115.400000
Name: Salary_LPA, dtype: float64

In [13]:
os.makedirs("../data/clean", exist_ok=True)
df.drop(columns=["Skills_List"]).to_csv("../data/clean/jobs_clean.csv", index=False)